In [1]:
from causalis.scenarios.unconfoundedness.dgp import generate_obs_hte_26_rich

data = generate_obs_hte_26_rich(return_causal_data=False, include_oracle=True)
data.head()

,user_id,y,d,tenure_months,avg_sessions_week,spend_last_month,age_years,income_monthly,prior_purchases_12m,support_tickets_90d,premium_user,mobile_user,urban_resident,referred_user,m,m_obs,tau_link,g0,g1,cate
0,1,0.000000,0.0,28.814654,1.0,77.936767,50.234101,1926.698301,1.0,2.0,1.0,1.0,1.0,0.0,0.045453,0.045453,0.089095,8.137981,9.142395,1.004414
1,2,80.099611,1.0,25.913345,3.0,53.777740,28.115859,5104.271509,3.0,0.0,1.0,1.0,0.0,1.0,0.041514,0.041514,0.246679,60.459257,78.817307,18.358049
2,3,6.400482,1.0,24.969929,10.0,134.764322,22.907062,5267.938255,8.0,3.0,0.0,1.0,1.0,0.0,0.052593,0.052593,0.162968,7.712855,9.138577,1.425723
3,4,2.788238,0.0,40.655089,5.0,59.517074,31.970490,6597.327018,3.0,2.0,1.0,1.0,1.0,0.0,0.036221,0.036221,0.188755,25.386510,31.159932,5.773422
4,5,0.000000,0.0,18.560899,3.0,74.370930,39.237248,4930.009628,5.0,1.0,1.0,1.0,0.0,0.0,0.036343,0.036343,0.174757,15.359250,18.600227,3.240977


In [2]:
print(f"Ground truth ATE is {data['cate'].mean()}")
print(f"Ground truth ATTE is {data[data['d'] == 1]['cate'].mean()}")

Ground truth ATE is 19.40958652966079
Ground truth ATTE is 10.914991423363862


In [3]:
from causalis.data_contracts import CausalData

causaldata = CausalData(df = data,
                        treatment='d',
                        user_id='user_id',
                        outcome='y',
                        confounders=['tenure_months',
                                     'avg_sessions_week',
                                     'spend_last_month',
                                     'age_years',
                                     'income_monthly',
                                     'prior_purchases_12m',
                                     'support_tickets_90d',
                                     'premium_user',
                                     'mobile_user',
                                     'urban_resident',
                                     'referred_user'])
causaldata

CausalData(df=(100000, 14), treatment='d', outcome='y', confounders=['tenure_months', 'avg_sessions_week', 'spend_last_month', 'age_years', 'income_monthly', 'prior_purchases_12m', 'support_tickets_90d', 'premium_user', 'mobile_user', 'urban_resident', 'referred_user'], user_id='user_id')

In [4]:
from causalis.scenarios.unconfoundedness import IRM

model = IRM(n_jobs=-1).fit(causaldata)

In [5]:
dml_result = model.estimate(score='ATTE')
dml_result.summary()

,value
field,
estimand,ATTE
model,IRM
value,"11.9234 (ci_abs: 7.2060, 16.6407)"
value_relative,"25.5961 (ci_rel: 15.1433, 36.0490)"
alpha,0.0500
p_value,0.0000
is_significant,True
n_treated,4949
n_control,95051


# Uplift / CATE scoring

`estimate(score='ATTE')` gives a historical average effect among treated clients. For targeting new clients we need CATE/uplift scores: expected treatment effect conditional on pre-treatment features.

The uplift scorer is lazy: `IRM.fit(...)`, `estimate(score='ATTE')`, and `estimate(score='GATE')` do not train extra scoring models. The first `predict_cate(...)` call trains final full-sample outcome models and caches them for reuse.

In [6]:
# Before uplift scoring, no final scoring models are stored on the IRM object.
(
    hasattr(model, '_uplift_g0_model_'),
    hasattr(model, '_uplift_g1_model_'),
)

(False, False)

## Score new clients

New scoring data must contain only pre-treatment features. In this synthetic notebook we sample rows from the generated data to mimic new clients, but we keep `cate` only as oracle ground truth for demonstration.

In [7]:
features = causaldata.confounders

df_new = (
    data[['user_id', *features, 'cate']]
    .sample(10, random_state=42)
    .reset_index(drop=True)
)

df_new[['user_id', *features]].head()

,user_id,tenure_months,avg_sessions_week,spend_last_month,age_years,income_monthly,prior_purchases_12m,support_tickets_90d,premium_user,mobile_user,urban_resident,referred_user
0,75722,28.139181,6.0,260.700949,27.125662,5931.570600,7.0,1.0,0.0,1.0,0.0,0.0
1,80185,11.185705,5.0,93.613706,18.000000,5548.751938,0.0,0.0,0.0,1.0,0.0,1.0
2,19865,14.081279,0.0,140.374257,40.455697,3075.923262,4.0,1.0,1.0,1.0,0.0,0.0
3,76700,4.301719,2.0,61.645222,18.000000,3760.445289,4.0,2.0,1.0,1.0,1.0,0.0
4,92992,36.365967,6.0,500.000000,47.953760,12208.911276,6.0,2.0,1.0,1.0,1.0,0.0


In [8]:
from causalis.scenarios.uplift import predict_cate

df_scored = df_new[['user_id']].copy()
df_scored['predicted_cate'] = predict_cate(model, df_new[features])
df_scored['oracle_cate'] = df_new['cate']
df_scored

,user_id,predicted_cate,oracle_cate
0,75722,-18.793821,1.927333
1,80185,-7.463641,1.192551
2,19865,52.915785,3.499984
3,76700,-5.524018,1.066981
4,92992,32.121917,17.968981
5,76435,25.408367,38.989121
6,84005,-298.488939,214.828824
7,80918,-6.028201,0.525502
8,60768,-12.203394,6.126396
9,50075,-3.288516,10.670732


In [9]:
# After the first scoring call, the final scoring models are cached and reused.
(
    hasattr(model, '_uplift_g0_model_'),
    hasattr(model, '_uplift_g1_model_'),
)

(True, True)

## Simple targeting policy

For a continuous LTV-style outcome, `predicted_cate` is already on the outcome scale. A simple policy targets clients when expected incremental LTV exceeds treatment cost.

In [10]:
treatment_cost = 0.05

df_scored['expected_incremental_value'] = df_scored['predicted_cate'] - treatment_cost
df_scored['recommended_treatment'] = df_scored['expected_incremental_value'] > 0

df_scored.sort_values('predicted_cate', ascending=False)

,user_id,predicted_cate,oracle_cate,expected_incremental_value,recommended_treatment
2,19865,52.915785,3.499984,52.865785,True
4,92992,32.121917,17.968981,32.071917,True
5,76435,25.408367,38.989121,25.358367,True
9,50075,-3.288516,10.670732,-3.338516,False
3,76700,-5.524018,1.066981,-5.574018,False
7,80918,-6.028201,0.525502,-6.078201,False
1,80185,-7.463641,1.192551,-7.513641,False
8,60768,-12.203394,6.126396,-12.253394,False
0,75722,-18.793821,1.927333,-18.843821,False
6,84005,-298.488939,214.828824,-298.538939,False


Interpretation: `predicted_cate` is not a true individual treatment effect. It is the model's estimate of the conditional average treatment effect for clients with similar pre-treatment features.